# Hybrid Lorentz-ParT Masked Autoencoder for JetClass: GSoC 2026 Implementation

A complete notebook implementation of a **hybrid ParT + Lorentz-aware** model for JetClass-like ROOT data with:
- masked autoencoder pretraining,
- supervised multi-class fine-tuning,
- optional mass regression,
- **attention-gated hybrid fusion** (proposed improvement),
- ablation-ready and interpretable diagnostics.


## SECTION 1 — INTRODUCTION

Jet/event classification is a core HEP problem: better discrimination means better sensitivity to rare physics and improved precision on known processes.

- **JetClass** provides realistic large-scale multi-class data for particle-cloud learning.
- **ParT** contributes set-based transformer modeling with pairwise interaction bias.
- **L-GATr** contributes Lorentz-aware reasoning and relativistic structure priors.
- Last year’s hybrid direction combined these ideas with masked reconstruction.

### Improvement in this notebook
We implement **attention-gated hybrid fusion** so the model learns whether to trust the ParT branch or the Lorentz-aware branch on each event/token, rather than relying on static fusion.

### Scope
This notebook includes:
- MAE-style masked pretraining,
- classification fine-tuning,
- optional mass regression,
- evaluation, ablations, and diagnostics.


## SECTION 2 — SETUP


In [ ]:
import os, re, math, time, json, random, warnings, copy
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

try:
    import uproot
    import awkward as ak
    HAS_ROOT = True
except Exception:
    HAS_ROOT = False


In [ ]:
@dataclass
class Config:
    """Global notebook config."""
    DATA_ROOT: str = './datasets/JetClass'
    MAX_PARTICLES: int = 128
    NUM_CLASSES: int = 10

    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 2
    PRETRAIN_EPOCHS: int = 3
    FINETUNE_EPOCHS: int = 5
    LEARNING_RATE: float = 2e-4
    WEIGHT_DECAY: float = 1e-4
    MASK_RATIO: float = 0.4

    USE_AUX_MASS: bool = True
    USE_MIXED_PRECISION: bool = True
    QUICK_DEMO_MODE: bool = True
    QUICK_MAX_EVENTS_PER_FILE: int = 3000

    NUM_TRAIN_FILES: Optional[int] = 20
    NUM_VAL_FILES: Optional[int] = 5
    NUM_TEST_FILES: Optional[int] = 5

    PER_PARTICLE_FEATURES: Tuple[str, ...] = (
        'px','py','pz','E','pt','eta','phi','charge','valid_mask'
    )
    PAIRWISE_FEATURES: Tuple[str, ...] = ('lnDelta','lnkT','lnz','lnm2')

    EMBED_DIM: int = 128
    NUM_HEADS: int = 8
    NUM_LAYERS_PART: int = 4
    NUM_LAYERS_LORENTZ: int = 3
    MLP_RATIO: float = 4.0
    DROPOUT: float = 0.1
    USE_TOKEN_GATE: bool = True
    USE_EVENT_GATE: bool = True

    W_RECO: float = 1.0
    W_CONS: float = 0.3
    W_CLASS: float = 1.0
    AUX_MASS_WEIGHT: float = 0.2

    GRAD_CLIP_NORM: float = 1.0
    SEED: int = 42

cfg = Config()
cfg


In [ ]:
def seed_everything(seed: int = 42) -> None:
    """Set seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_enabled = cfg.USE_MIXED_PRECISION and device.type == 'cuda'
print('Device:', device)
print('Mixed precision:', amp_enabled)
print('ROOT stack available:', HAS_ROOT)


## SECTION 3 — DATA UNDERSTANDING


In [ ]:
def discover_root_files(data_root: Path) -> List[Path]:
    """Recursively discover .root files."""
    return sorted(data_root.rglob('*.root')) if data_root.exists() else []

def infer_class_name(path: Path) -> str:
    """Infer class from filename prefix."""
    return path.stem.split('_')[0]

data_root = Path(cfg.DATA_ROOT)
files = discover_root_files(data_root)
print('Data root:', data_root.resolve())
print('ROOT files found:', len(files))
for p in files[:8]:
    print(' -', p)

classes = sorted({infer_class_name(p) for p in files}) if files else []
print('Inferred classes:', classes)


In [ ]:
BRANCH_CANDIDATES = {
    'px': ['part_px','px','PFCands_px','particle_px'],
    'py': ['part_py','py','PFCands_py','particle_py'],
    'pz': ['part_pz','pz','PFCands_pz','particle_pz'],
    'E': ['part_energy','part_e','energy','E','PFCands_energy','particle_e'],
    'pt': ['part_pt','pt','PFCands_pt','particle_pt'],
    'eta': ['part_eta','eta','PFCands_eta','particle_eta'],
    'phi': ['part_phi','phi','PFCands_phi','particle_phi'],
    'charge': ['part_charge','charge','PFCands_charge','particle_charge'],
}

def resolve_branches(available: List[str], candidates: Dict[str, List[str]]) -> Dict[str, Optional[str]]:
    """Resolve semantic names to existing branches."""
    lower = {a.lower(): a for a in available}
    out = {}
    for k, opts in candidates.items():
        hit = None
        for o in opts:
            if o in available:
                hit = o
                break
            if o.lower() in lower:
                hit = lower[o.lower()]
                break
        out[k] = hit
    return out

if HAS_ROOT and files:
    with uproot.open(files[0]) as f:
        tree_name = None
        for c in ['tree','events','Events','JetTree']:
            if c in f:
                tree_name = c
                break
        if tree_name is None:
            for k in f.keys():
                if hasattr(f[k], 'keys'):
                    tree_name = k
                    break
        print('Selected tree:', tree_name)
        branches = [str(b) for b in f[tree_name].keys()]
        print('First branches:', branches[:30])
        print('Resolved map:', resolve_branches(branches, BRANCH_CANDIDATES))
else:
    print('No ROOT file available for branch inspection.')


## SECTION 4 — ROOT DATA LOADING


In [ ]:
def safe_eta(px: np.ndarray, py: np.ndarray, pz: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Compute pseudorapidity safely."""
    p = np.sqrt(px**2 + py**2 + pz**2 + eps)
    return 0.5 * np.log((p + pz + eps) / (p - pz + eps))

def read_jetclass_root(file_path: Path, max_events: Optional[int]) -> List[Dict[str, np.ndarray]]:
    """Read one ROOT file into per-event particle arrays."""
    if not HAS_ROOT:
        raise RuntimeError('uproot/awkward not installed.')

    events = []
    with uproot.open(file_path) as f:
        tree_name = None
        for c in ['tree','events','Events','JetTree']:
            if c in f:
                tree_name = c
                break
        if tree_name is None:
            for k in f.keys():
                if hasattr(f[k], 'arrays'):
                    tree_name = k
                    break
        tree = f[tree_name]
        av = [str(x) for x in tree.keys()]
        bm = resolve_branches(av, BRANCH_CANDIDATES)

        if bm['px'] is None or bm['py'] is None or bm['pz'] is None:
            raise ValueError(f'Missing required momentum branches in {file_path.name}')

        read_keys = [v for v in bm.values() if v is not None]
        arrs = tree.arrays(read_keys, library='ak', entry_stop=max_events)

        pxL = [np.asarray(x, dtype=np.float32) for x in ak.to_list(arrs[bm['px']])]
        pyL = [np.asarray(x, dtype=np.float32) for x in ak.to_list(arrs[bm['py']])]
        pzL = [np.asarray(x, dtype=np.float32) for x in ak.to_list(arrs[bm['pz']])]

        if bm['E'] is not None:
            EL = [np.asarray(x, dtype=np.float32) for x in ak.to_list(arrs[bm['E']])]
        else:
            EL = [np.sqrt(px**2 + py**2 + pz**2 + 1e-8).astype(np.float32) for px,py,pz in zip(pxL,pyL,pzL)]

        if bm['charge'] is not None:
            qL = [np.asarray(x, dtype=np.float32) for x in ak.to_list(arrs[bm['charge']])]
        else:
            qL = [np.zeros_like(px, dtype=np.float32) for px in pxL]

        for px, py, pz, E, q in zip(pxL, pyL, pzL, EL, qL):
            pt = np.sqrt(px**2 + py**2 + 1e-8).astype(np.float32)
            eta = safe_eta(px, py, pz).astype(np.float32)
            phi = np.arctan2(py, px).astype(np.float32)
            events.append({
                'px': px, 'py': py, 'pz': pz, 'E': E,
                'pt': pt, 'eta': eta, 'phi': phi,
                'charge': q, 'valid_mask': np.ones_like(px, dtype=np.float32)
            })
    return events

def pad_event(ev: Dict[str,np.ndarray], max_particles: int, feat_order: Tuple[str,...]):
    """Pad/truncate event to fixed length and return (x,mask,p4)."""
    x = np.zeros((max_particles, len(feat_order)), dtype=np.float32)
    mask = np.zeros((max_particles,), dtype=np.float32)
    p4 = np.zeros((max_particles, 4), dtype=np.float32)

    n = min(len(ev['px']), max_particles)
    mask[:n] = 1.0

    for j, name in enumerate(feat_order):
        arr = np.nan_to_num(ev.get(name, np.zeros_like(ev['px'])), nan=0.0, posinf=0.0, neginf=0.0)
        x[:n, j] = arr[:n]

    p4[:n, 0] = np.nan_to_num(ev['E'][:n], nan=0.0, posinf=0.0, neginf=0.0)
    p4[:n, 1] = np.nan_to_num(ev['px'][:n], nan=0.0, posinf=0.0, neginf=0.0)
    p4[:n, 2] = np.nan_to_num(ev['py'][:n], nan=0.0, posinf=0.0, neginf=0.0)
    p4[:n, 3] = np.nan_to_num(ev['pz'][:n], nan=0.0, posinf=0.0, neginf=0.0)
    return x, mask, p4

def event_mass(p4: np.ndarray, mask: np.ndarray) -> float:
    """Invariant mass of event-level four-vector sum."""
    vv = p4[mask > 0.5]
    if vv.size == 0:
        return 0.0
    s = vv.sum(axis=0)
    m2 = s[0]**2 - (s[1]**2 + s[2]**2 + s[3]**2)
    return float(np.sqrt(max(m2, 0.0)))


In [ ]:
class JetClassDataset(Dataset):
    """Padded JetClass dataset."""
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, mask, p4, y, mass = self.samples[idx]
        return {
            'x': torch.tensor(x, dtype=torch.float32),
            'mask': torch.tensor(mask, dtype=torch.float32),
            'p4': torch.tensor(p4, dtype=torch.float32),
            'y': torch.tensor(y, dtype=torch.long),
            'mass': torch.tensor(mass, dtype=torch.float32),
        }

def fit_scaler(samples) -> StandardScaler:
    """Fit scaler on valid train particles."""
    scaler = StandardScaler()
    buf = []
    for x, mask, *_ in samples:
        v = x[mask > 0.5]
        if len(v):
            buf.append(v)
    if buf:
        scaler.fit(np.concatenate(buf, axis=0))
    else:
        scaler.fit(np.zeros((1, len(cfg.PER_PARTICLE_FEATURES)), dtype=np.float32))
    return scaler

def apply_scaler(samples, scaler):
    """Apply scaler on valid particles only."""
    out = []
    for x, mask, p4, y, m in samples:
        x2 = x.copy()
        if (mask > 0.5).any():
            x2[mask > 0.5] = scaler.transform(x2[mask > 0.5])
        out.append((x2, mask, p4, y, m))
    return out


In [ ]:
def build_loaders(cfg: Config):
    """Build DataLoaders from ROOT files, with synthetic fallback."""
    files = discover_root_files(Path(cfg.DATA_ROOT))

    if files:
        cls = sorted({infer_class_name(f) for f in files})
        cfg.NUM_CLASSES = len(cls)
        c2i = {c:i for i,c in enumerate(cls)}
        labels = [infer_class_name(f) for f in files]

        tr, tmp, ytr, ytmp = train_test_split(files, labels, test_size=0.2, random_state=cfg.SEED, stratify=labels)
        va, te = train_test_split(tmp, test_size=0.5, random_state=cfg.SEED, stratify=ytmp)
        if cfg.NUM_TRAIN_FILES is not None: tr = tr[:cfg.NUM_TRAIN_FILES]
        if cfg.NUM_VAL_FILES is not None: va = va[:cfg.NUM_VAL_FILES]
        if cfg.NUM_TEST_FILES is not None: te = te[:cfg.NUM_TEST_FILES]

        def load_subset(subfiles):
            samples = []
            max_events = cfg.QUICK_MAX_EVENTS_PER_FILE if cfg.QUICK_DEMO_MODE else None
            for fp in tqdm(subfiles, desc='Reading ROOT'):
                y = c2i[infer_class_name(fp)]
                try:
                    events = read_jetclass_root(fp, max_events)
                except Exception as e:
                    print('[WARN] skip', fp.name, e)
                    continue
                for ev in events:
                    x, m, p4 = pad_event(ev, cfg.MAX_PARTICLES, cfg.PER_PARTICLE_FEATURES)
                    samples.append((x, m, p4, y, event_mass(p4, m)))
            return samples

        trS, vaS, teS = load_subset(tr), load_subset(va), load_subset(te)
        scaler = fit_scaler(trS)
        trS, vaS, teS = apply_scaler(trS, scaler), apply_scaler(vaS, scaler), apply_scaler(teS, scaler)

    else:
        print('[INFO] ROOT files not found; using synthetic fallback demo data.')
        cls = [f'Class{i}' for i in range(cfg.NUM_CLASSES)]

        def synth(n_events):
            out = []
            for _ in range(n_events):
                n = np.random.randint(8, cfg.MAX_PARTICLES)
                px = np.random.normal(0, 40, n).astype(np.float32)
                py = np.random.normal(0, 40, n).astype(np.float32)
                pz = np.random.normal(0, 60, n).astype(np.float32)
                E = np.sqrt(px**2 + py**2 + pz**2 + np.random.uniform(0, 4, n)**2).astype(np.float32)
                ev = {
                    'px': px, 'py': py, 'pz': pz, 'E': E,
                    'pt': np.sqrt(px**2 + py**2 + 1e-8).astype(np.float32),
                    'eta': safe_eta(px, py, pz).astype(np.float32),
                    'phi': np.arctan2(py, px).astype(np.float32),
                    'charge': np.random.choice([-1, 0, 1], size=n).astype(np.float32),
                    'valid_mask': np.ones(n, dtype=np.float32),
                }
                x, m, p4 = pad_event(ev, cfg.MAX_PARTICLES, cfg.PER_PARTICLE_FEATURES)
                y = np.random.randint(0, cfg.NUM_CLASSES)
                out.append((x, m, p4, y, event_mass(p4, m)))
            return out

        trS, vaS, teS = synth(3000), synth(600), synth(600)
        scaler = fit_scaler(trS)
        trS, vaS, teS = apply_scaler(trS, scaler), apply_scaler(vaS, scaler), apply_scaler(teS, scaler)

    trL = DataLoader(JetClassDataset(trS), batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS)
    vaL = DataLoader(JetClassDataset(vaS), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)
    teL = DataLoader(JetClassDataset(teS), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS)
    return trL, vaL, teL, {'classes': cls}


## SECTION 5 — FEATURE ENGINEERING

Particle features include cartesian and cylindrical kinematics and simple identity-like channels:
- `px, py, pz, E`
- `pt, eta, phi`
- `charge`
- `valid_mask`

Pairwise ParT-style features implemented below:
- `ln(Delta)` angular separation,
- `ln(kT)` relative transverse scale,
- `ln(z)` momentum sharing,
- `ln(m^2)` pairwise invariant mass proxy.

Numerical safeguards (`eps`, clamping, NaN/Inf cleanup) are included.


In [ ]:
def wrap_dphi(dphi: torch.Tensor) -> torch.Tensor:
    """Wrap angle to [-pi, pi]."""
    return (dphi + math.pi) % (2 * math.pi) - math.pi

def compute_pairwise_features(x: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Compute ParT-inspired pairwise features [B,N,N,4]."""
    px, py, pz, E = x[...,0], x[...,1], x[...,2], x[...,3]
    pt = x[...,4].abs() + eps
    eta, phi = x[...,5], x[...,6]

    d_eta = eta.unsqueeze(2) - eta.unsqueeze(1)
    d_phi = wrap_dphi(phi.unsqueeze(2) - phi.unsqueeze(1))
    delta = torch.sqrt(d_eta**2 + d_phi**2 + eps)
    lnDelta = torch.log(delta + eps)

    pti, ptj = pt.unsqueeze(2), pt.unsqueeze(1)
    kT = torch.minimum(pti, ptj) * delta
    lnkT = torch.log(kT + eps)

    z = torch.minimum(pti, ptj) / (pti + ptj + eps)
    lnz = torch.log(z + eps)

    Ei, Ej = E.unsqueeze(2), E.unsqueeze(1)
    pxi, pxj = px.unsqueeze(2), px.unsqueeze(1)
    pyi, pyj = py.unsqueeze(2), py.unsqueeze(1)
    pzi, pzj = pz.unsqueeze(2), pz.unsqueeze(1)
    m2 = (Ei+Ej)**2 - ((pxi+pxj)**2 + (pyi+pyj)**2 + (pzi+pzj)**2)
    lnm2 = torch.log(torch.clamp(m2.abs(), min=eps))

    pair = torch.stack([lnDelta, lnkT, lnz, lnm2], dim=-1)
    valid = (mask.unsqueeze(2) * mask.unsqueeze(1)).unsqueeze(-1)
    return torch.nan_to_num(pair * valid, nan=0.0, posinf=0.0, neginf=0.0)


## SECTION 6 — MASKED AUTOENCODER PRETRAINING DATA PIPELINE


In [ ]:
def make_random_mask(valid_mask: torch.Tensor, mask_ratio: float) -> torch.Tensor:
    """Randomly mask valid particles only. True = masked token."""
    B, N = valid_mask.shape
    m = torch.zeros((B, N), dtype=torch.bool, device=valid_mask.device)
    for b in range(B):
        idx = torch.where(valid_mask[b] > 0.5)[0]
        if len(idx) == 0:
            continue
        k = max(1, int(len(idx) * mask_ratio))
        perm = torch.randperm(len(idx), device=valid_mask.device)
        m[b, idx[perm[:k]]] = True
    return m

def make_structured_mask_highpt(x: torch.Tensor, valid_mask: torch.Tensor, frac: float = 0.2) -> torch.Tensor:
    """Structured masking: mask highest-pt particles."""
    pt = x[...,4]
    B, N = pt.shape
    m = torch.zeros((B, N), dtype=torch.bool, device=x.device)
    for b in range(B):
        idx = torch.where(valid_mask[b] > 0.5)[0]
        if len(idx) == 0:
            continue
        k = max(1, int(len(idx) * frac))
        order = torch.argsort(pt[b, idx], descending=True)
        m[b, idx[order[:k]]] = True
    return m

def build_mae_targets(x: torch.Tensor, p4: torch.Tensor, valid_mask: torch.Tensor, mask_ratio: float, structured: bool = False):
    """Build masked inputs and reconstruction targets."""
    reco_mask = make_structured_mask_highpt(x, valid_mask, mask_ratio) if structured else make_random_mask(valid_mask, mask_ratio)
    visible_x = x.clone(); visible_x[reco_mask] = 0.0
    reco_target = x.clone()

    masked_p4 = p4.clone(); masked_p4[~reco_mask] = 0.0
    conservation_target = masked_p4.sum(dim=1)
    return visible_x, reco_target, reco_mask, conservation_target


## SECTION 7 — MODEL DESIGN


In [ ]:
class ParticleFeatureEmbed(nn.Module):
    """Embed per-particle features."""
    def __init__(self, in_dim: int, d: int, drop: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, d))
    def forward(self, x):
        return self.net(x)

class PairwiseFeatureEncoder(nn.Module):
    """Map pairwise features -> attention bias per head."""
    def __init__(self, in_dim: int, heads: int):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, 32), nn.GELU(), nn.Linear(32, heads))
    def forward(self, pair):
        return self.proj(pair).permute(0,3,1,2).contiguous()


In [ ]:
class ParTAttentionBlock(nn.Module):
    """ParT block with pairwise attention bias."""
    def __init__(self, d: int, heads: int, mlp_ratio: float = 4.0, drop: float = 0.1):
        super().__init__()
        assert d % heads == 0
        self.h, self.dh = heads, d // heads
        self.scale = self.dh ** -0.5
        self.n1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3*d)
        self.out = nn.Linear(d, d)
        self.n2 = nn.LayerNorm(d)
        hidden = int(d * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, d))
        self.drop = nn.Dropout(drop)

    def forward(self, x, mask, pair_bias):
        B, N, D = x.shape
        qkv = self.qkv(self.n1(x)).reshape(B, N, 3, self.h, self.dh)
        q, k, v = qkv[:,:,0].permute(0,2,1,3), qkv[:,:,1].permute(0,2,1,3), qkv[:,:,2].permute(0,2,1,3)

        attn = (q @ k.transpose(-2,-1)) * self.scale + pair_bias
        km = (mask.unsqueeze(1).unsqueeze(2) > 0.5)
        attn = attn.masked_fill(~km, -1e9)
        attn = torch.softmax(attn, dim=-1)

        o = (attn @ v).permute(0,2,1,3).reshape(B, N, D)
        x = x + self.drop(self.out(o))
        x = x + self.drop(self.mlp(self.n2(x)))
        return x * mask.unsqueeze(-1)


In [ ]:
class LightweightLorentzBlock(nn.Module):
    """L-GATr-inspired lightweight Lorentz-aware token mixer.

    Approximation note: this is not full geometric-algebra L-GATr;
    it uses four-vector invariants and Minkowski inner-product biases.
    """
    def __init__(self, d: int, heads: int, mlp_ratio: float = 4.0, drop: float = 0.1):
        super().__init__()
        assert d % heads == 0
        self.h, self.dh = heads, d // heads
        self.scale = self.dh ** -0.5
        self.n1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3*d)
        self.out = nn.Linear(d, d)

        self.inv_proj = nn.Sequential(nn.Linear(3, 32), nn.GELU(), nn.Linear(32, heads))

        self.n2 = nn.LayerNorm(d)
        hidden = int(d * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, d))
        self.drop = nn.Dropout(drop)

    @staticmethod
    def mdot(a, b):
        return a[...,0]*b[...,0] - (a[...,1]*b[...,1] + a[...,2]*b[...,2] + a[...,3]*b[...,3])

    def lorentz_bias(self, p4, mask, eps: float = 1e-8):
        pi, pj = p4.unsqueeze(2), p4.unsqueeze(1)
        md = self.mdot(pi, pj)
        mi2 = self.mdot(pi, pi)
        mj2 = self.mdot(pj, pj)
        sij = self.mdot(pi+pj, pi+pj)

        feats = torch.stack([
            torch.log(torch.clamp(md.abs(), min=eps)),
            torch.log(torch.clamp(0.5*(mi2.abs()+mj2.abs()), min=eps)),
            torch.log(torch.clamp(sij.abs(), min=eps)),
        ], dim=-1)
        bias = self.inv_proj(feats).permute(0,3,1,2).contiguous()
        valid = (mask.unsqueeze(2)*mask.unsqueeze(1)).unsqueeze(1)
        return bias * valid

    def forward(self, x, p4, mask):
        B, N, D = x.shape
        qkv = self.qkv(self.n1(x)).reshape(B, N, 3, self.h, self.dh)
        q, k, v = qkv[:,:,0].permute(0,2,1,3), qkv[:,:,1].permute(0,2,1,3), qkv[:,:,2].permute(0,2,1,3)
        attn = (q @ k.transpose(-2,-1)) * self.scale + self.lorentz_bias(p4, mask)
        km = (mask.unsqueeze(1).unsqueeze(2) > 0.5)
        attn = attn.masked_fill(~km, -1e9)
        attn = torch.softmax(attn, dim=-1)

        o = (attn @ v).permute(0,2,1,3).reshape(B,N,D)
        x = x + self.drop(self.out(o))
        x = x + self.drop(self.mlp(self.n2(x)))
        return x * mask.unsqueeze(-1)


In [ ]:
class HybridGatedFusion(nn.Module):
    """Learned fusion gate between ParT and Lorentz branches."""
    def __init__(self, d: int, token_gate: bool = True, event_gate: bool = True):
        super().__init__()
        self.token_gate = token_gate
        self.event_gate = event_gate
        if token_gate:
            self.gtok = nn.Sequential(nn.Linear(2*d, d), nn.GELU(), nn.Linear(d, d), nn.Sigmoid())
        if event_gate:
            self.gevt = nn.Sequential(nn.Linear(2*d, d//2), nn.GELU(), nn.Linear(d//2, 1), nn.Sigmoid())

    @staticmethod
    def masked_mean(x, mask):
        den = mask.sum(dim=1, keepdim=True).clamp_min(1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / den

    def forward(self, xp, xl, mask):
        c = torch.cat([xp, xl], dim=-1)
        gtok = self.gtok(c) if self.token_gate else torch.full_like(xp, 0.5)
        fused = gtok * xp + (1-gtok) * xl

        if self.event_gate:
            ep = self.masked_mean(xp, mask)
            el = self.masked_mean(xl, mask)
            ge = self.gevt(torch.cat([ep, el], dim=-1))
            fused = ge.unsqueeze(-1)*fused + (1-ge.unsqueeze(-1))*0.5*(xp+xl)
        else:
            ge = torch.full((xp.shape[0],1), 0.5, device=xp.device)

        return fused * mask.unsqueeze(-1), gtok, ge


In [ ]:
class MaskedDecoderHead(nn.Module):
    """Decode masked particle features."""
    def __init__(self, d: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,d), nn.GELU(), nn.Linear(d,out_dim))
    def forward(self, x):
        return self.net(x)

class ClassificationHead(nn.Module):
    """Event-level classifier."""
    def __init__(self, d: int, ncls: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d,ncls))
    def forward(self, z):
        return self.net(z)

class OptionalMassRegressionHead(nn.Module):
    """Optional event mass regressor."""
    def __init__(self, d: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,d//2), nn.GELU(), nn.Linear(d//2,1))
    def forward(self, z):
        return self.net(z).squeeze(-1)


In [ ]:
class HybridLorentzParTMAE(nn.Module):
    """Full hybrid model with pretrain/finetune/multitask modes."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        fin = len(cfg.PER_PARTICLE_FEATURES)
        d = cfg.EMBED_DIM

        self.embed = ParticleFeatureEmbed(fin, d, cfg.DROPOUT)
        self.penc = PairwiseFeatureEncoder(len(cfg.PAIRWISE_FEATURES), cfg.NUM_HEADS)

        self.part = nn.ModuleList([ParTAttentionBlock(d, cfg.NUM_HEADS, cfg.MLP_RATIO, cfg.DROPOUT) for _ in range(max(cfg.NUM_LAYERS_PART, 1))])
        self.lor = nn.ModuleList([LightweightLorentzBlock(d, cfg.NUM_HEADS, cfg.MLP_RATIO, cfg.DROPOUT) for _ in range(max(cfg.NUM_LAYERS_LORENTZ, 1))])

        self.fusion = HybridGatedFusion(d, cfg.USE_TOKEN_GATE, cfg.USE_EVENT_GATE)
        self.dec = MaskedDecoderHead(d, fin)
        self.cls = ClassificationHead(d, cfg.NUM_CLASSES)
        self.mass = OptionalMassRegressionHead(d)

    @staticmethod
    def masked_mean(x, mask):
        den = mask.sum(dim=1, keepdim=True).clamp_min(1.0)
        return (x * mask.unsqueeze(-1)).sum(dim=1) / den

    def encode(self, x, p4, mask):
        h0 = self.embed(x) * mask.unsqueeze(-1)
        pair = compute_pairwise_features(x, mask)
        pbias = self.penc(pair)

        hp = h0
        if self.cfg.NUM_LAYERS_PART > 0:
            for blk in self.part[:self.cfg.NUM_LAYERS_PART]:
                hp = blk(hp, mask, pbias)

        hl = h0
        if self.cfg.NUM_LAYERS_LORENTZ > 0:
            for blk in self.lor[:self.cfg.NUM_LAYERS_LORENTZ]:
                hl = blk(hl, p4, mask)

        return hp, hl

    def forward(self, x, p4, mask, mode: str = 'finetune'):
        hp, hl = self.encode(x, p4, mask)
        hf, gtok, gevt = self.fusion(hp, hl, mask)
        pooled = self.masked_mean(hf, mask)

        out = {
            'logits': self.cls(pooled),
            'mass_pred': self.mass(pooled),
            'reco': self.dec(hf),
            'g_tok': gtok,
            'g_evt': gevt,
        }
        if mode == 'pretrain':
            return {'reco': out['reco'], 'g_tok': out['g_tok'], 'g_evt': out['g_evt']}
        if mode == 'finetune':
            return {'logits': out['logits'], 'g_tok': out['g_tok'], 'g_evt': out['g_evt']}
        if mode == 'multitask':
            return out
        raise ValueError(f'Unknown mode {mode}')


## SECTION 8 — LOSS FUNCTIONS


In [ ]:
def masked_reco_loss(reco, target, reco_mask):
    """MSE over masked tokens only."""
    w = reco_mask.unsqueeze(-1).float()
    return (((reco-target)**2)*w).sum() / w.sum().clamp_min(1.0)

def conservation_loss(reco, target, reco_mask, feat_idx):
    """Compare masked momentum sums and masked invariant-mass proxy."""
    iE, ipx, ipy, ipz = feat_idx['E'], feat_idx['px'], feat_idx['py'], feat_idx['pz']
    m = reco_mask.float()

    pred = torch.stack([
        (reco[...,iE] * m).sum(1),
        (reco[...,ipx] * m).sum(1),
        (reco[...,ipy] * m).sum(1),
        (reco[...,ipz] * m).sum(1),
    ], dim=-1)
    true = torch.stack([
        (target[...,iE] * m).sum(1),
        (target[...,ipx] * m).sum(1),
        (target[...,ipy] * m).sum(1),
        (target[...,ipz] * m).sum(1),
    ], dim=-1)

    lp4 = F.smooth_l1_loss(pred, true)
    pm2 = pred[:,0]**2 - (pred[:,1]**2 + pred[:,2]**2 + pred[:,3]**2)
    tm2 = true[:,0]**2 - (true[:,1]**2 + true[:,2]**2 + true[:,3]**2)
    lm = F.smooth_l1_loss(torch.clamp(pm2, min=0.0), torch.clamp(tm2, min=0.0))
    return lp4 + 0.5*lm

class FocalLoss(nn.Module):
    """Optional focal loss helper."""
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, reduction='none')
        pt = torch.exp(-ce)
        return (((1-pt)**self.gamma) * ce).mean()

def pretrain_total_loss(out, target_x, reco_mask, cfg, feat_idx):
    """Weighted pretraining loss."""
    lr = masked_reco_loss(out['reco'], target_x, reco_mask)
    lc = conservation_loss(out['reco'], target_x, reco_mask, feat_idx)
    return cfg.W_RECO*lr + cfg.W_CONS*lc, {'reco': lr.item(), 'cons': lc.item()}

def finetune_total_loss(out, batch, cfg, focal=False):
    """Weighted finetune/multitask loss."""
    lcls = FocalLoss()(out['logits'], batch['y']) if focal else F.cross_entropy(out['logits'], batch['y'])
    total = cfg.W_CLASS * lcls
    logs = {'cls': lcls.item()}
    if cfg.USE_AUX_MASS and 'mass_pred' in out:
        lm = F.smooth_l1_loss(out['mass_pred'], batch['mass'])
        total = total + cfg.AUX_MASS_WEIGHT * lm
        logs['mass'] = lm.item()
    return total, logs


## SECTION 9 — TRAINING UTILITIES


In [ ]:
class AverageMeter:
    """Running average meter."""
    def __init__(self): self.reset()
    def reset(self): self.sum=0.0; self.n=0
    def update(self, v, k=1): self.sum += float(v)*k; self.n += k
    @property
    def avg(self): return self.sum / max(self.n, 1)

class EarlyStopping:
    """Simple early stopping utility."""
    def __init__(self, patience=5, mode='min'):
        self.patience=patience; self.mode=mode; self.best=None; self.bad=0
    def step(self, x):
        if self.best is None:
            self.best=x; return False
        imp = x < self.best if self.mode=='min' else x > self.best
        if imp: self.best=x; self.bad=0
        else: self.bad += 1
        return self.bad >= self.patience

def make_optimizer(model, cfg):
    """Build AdamW optimizer."""
    return torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)

def make_scheduler(opt):
    """Build cosine scheduler."""
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)

def save_ckpt(path: Path, model, opt, sch, epoch, best):
    """Save checkpoint."""
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'model': model.state_dict(), 'optimizer': opt.state_dict(), 'scheduler': sch.state_dict() if sch else None, 'epoch': epoch, 'best': best}, path)

def load_ckpt(path: Path, model, opt=None, sch=None):
    """Load checkpoint."""
    ck = torch.load(path, map_location='cpu')
    model.load_state_dict(ck['model'], strict=False)
    if opt is not None and ck.get('optimizer') is not None: opt.load_state_dict(ck['optimizer'])
    if sch is not None and ck.get('scheduler') is not None: sch.load_state_dict(ck['scheduler'])
    return ck

def to_device(batch, device):
    """Move batch to target device."""
    return {k: v.to(device, non_blocking=True) for k,v in batch.items()}


In [ ]:
def train_epoch_pretrain(model, loader, opt, scaler, device, cfg, feat_idx):
    """One pretraining epoch."""
    model.train()
    lm, lr, lc = AverageMeter(), AverageMeter(), AverageMeter()
    t0 = time.time()

    for batch in tqdm(loader, desc='Pretrain', leave=False):
        batch = to_device(batch, device)
        vis_x, target_x, reco_mask, _ = build_mae_targets(batch['x'], batch['p4'], batch['mask'], cfg.MASK_RATIO)
        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            out = model(vis_x, batch['p4'], batch['mask'], mode='pretrain')
            loss, logs = pretrain_total_loss(out, target_x, reco_mask, cfg, feat_idx)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
        scaler.step(opt); scaler.update()

        bsz = batch['x'].size(0)
        lm.update(loss.item(), bsz)
        lr.update(logs['reco'], bsz)
        lc.update(logs['cons'], bsz)

    return {'loss': lm.avg, 'reco': lr.avg, 'cons': lc.avg, 'time': time.time()-t0}

def train_epoch_finetune(model, loader, opt, scaler, device, cfg):
    """One fine-tuning epoch."""
    model.train()
    lm, am = AverageMeter(), AverageMeter()
    t0 = time.time()

    for batch in tqdm(loader, desc='Finetune', leave=False):
        batch = to_device(batch, device)
        opt.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            out = model(batch['x'], batch['p4'], batch['mask'], mode='multitask' if cfg.USE_AUX_MASS else 'finetune')
            loss, _ = finetune_total_loss(out, batch, cfg)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
        scaler.step(opt); scaler.update()

        pred = out['logits'].argmax(1)
        acc = (pred == batch['y']).float().mean().item()
        bsz = batch['x'].size(0)
        lm.update(loss.item(), bsz); am.update(acc, bsz)

    return {'loss': lm.avg, 'acc': am.avg, 'time': time.time()-t0}

@torch.no_grad()
def eval_finetune(model, loader, device, cfg):
    """Evaluate classifier and optional mass regressor."""
    model.eval()
    Y, P, YH, MT, MP = [], [], [], [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        batch = to_device(batch, device)
        out = model(batch['x'], batch['p4'], batch['mask'], mode='multitask' if cfg.USE_AUX_MASS else 'finetune')
        prob = torch.softmax(out['logits'], dim=1)
        yhat = prob.argmax(1)
        Y.append(batch['y'].cpu().numpy()); P.append(prob.cpu().numpy()); YH.append(yhat.cpu().numpy())
        MT.append(batch['mass'].cpu().numpy())
        if 'mass_pred' in out: MP.append(out['mass_pred'].cpu().numpy())

    y = np.concatenate(Y); p = np.concatenate(P); yh = np.concatenate(YH)
    met = {'accuracy': accuracy_score(y, yh)}
    try:
        met['macro_auc_ovr'] = roc_auc_score(y, p, multi_class='ovr', average='macro')
    except Exception:
        met['macro_auc_ovr'] = np.nan

    if len(MP):
        mt, mp = np.concatenate(MT), np.concatenate(MP)
        met['mass_mae'] = mean_absolute_error(mt, mp)
        met['mass_rmse'] = mean_squared_error(mt, mp, squared=False)

    return met, y, p, yh


## SECTION 10 — EVALUATION METRICS


In [ ]:
def plot_curves(hist: Dict[str, List[float]], title: str):
    """Plot metric curves."""
    plt.figure(figsize=(8,4))
    for k,v in hist.items():
        plt.plot(v, label=k)
    plt.title(title); plt.xlabel('Epoch'); plt.ylabel('Value')
    plt.grid(alpha=0.3); plt.legend(); plt.show()

def plot_confusion(y, yh, class_names):
    """Plot confusion matrix."""
    cm = confusion_matrix(y, yh)
    fig, ax = plt.subplots(figsize=(7,6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title('Confusion Matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticks(np.arange(len(class_names))); ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right'); ax.set_yticklabels(class_names)
    fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

def plot_rocs(y, p, class_names):
    """Plot per-class one-vs-rest ROC curves."""
    C = p.shape[1]
    yoh = np.eye(C)[y]
    plt.figure(figsize=(8,6))
    for i in range(C):
        try:
            fpr, tpr, _ = roc_curve(yoh[:,i], p[:,i])
            plt.plot(fpr, tpr, label=f'{class_names[i]} AUC={auc(fpr,tpr):.3f}')
        except Exception:
            pass
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('Per-class ROC')
    plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.show()

def bkg_rej_at_sig_eff(y, p, sig_class=0, eff=0.5):
    """Compute background rejection at fixed signal efficiency."""
    yb = (y == sig_class).astype(int)
    fpr, tpr, _ = roc_curve(yb, p[:,sig_class])
    i = np.argmin(np.abs(tpr-eff))
    return float(1.0 / max(fpr[i], 1e-8))


## SECTION 11 — PRETRAINING RUN


In [ ]:
train_loader, val_loader, test_loader, meta = build_loaders(cfg)
class_names = meta['classes']
print('Classes:', class_names)
print('Batches train/val/test:', len(train_loader), len(val_loader), len(test_loader))

model = HybridLorentzParTMAE(cfg).to(device)
opt = make_optimizer(model, cfg)
sch = make_scheduler(opt)
scaler = torch.amp.GradScaler(device=device.type, enabled=amp_enabled)
feat_idx = {k:i for i,k in enumerate(cfg.PER_PARTICLE_FEATURES)}

pre_hist = {'loss': [], 'reco': [], 'cons': []}
for ep in range(cfg.PRETRAIN_EPOCHS):
    logs = train_epoch_pretrain(model, train_loader, opt, scaler, device, cfg, feat_idx)
    sch.step()
    pre_hist['loss'].append(logs['loss'])
    pre_hist['reco'].append(logs['reco'])
    pre_hist['cons'].append(logs['cons'])
    print(f"[Pretrain {ep+1}/{cfg.PRETRAIN_EPOCHS}] loss={logs['loss']:.4f} reco={logs['reco']:.4f} cons={logs['cons']:.4f} t={logs['time']:.1f}s")


In [ ]:
plot_curves(pre_hist, 'Pretraining Loss Curves')


In [ ]:
@torch.no_grad()
def show_reco_examples(model, loader, cfg, n_events=2):
    """Print reconstructed vs true features for masked particles."""
    model.eval()
    batch = next(iter(loader)); batch = to_device(batch, device)
    vis, tgt, rm, _ = build_mae_targets(batch['x'], batch['p4'], batch['mask'], cfg.MASK_RATIO)
    out = model(vis, batch['p4'], batch['mask'], mode='pretrain')

    names = list(cfg.PER_PARTICLE_FEATURES)
    for b in range(min(n_events, batch['x'].size(0))):
        idx = torch.where(rm[b])[0][:8]
        if len(idx) == 0: continue
        print(f'Event {b}, masked tokens shown: {len(idx)}')
        for fname in ['px','py','pz','E','pt']:
            j = names.index(fname)
            t = tgt[b, idx, j].detach().cpu().numpy()
            p = out['reco'][b, idx, j].detach().cpu().numpy()
            print(f'  {fname} true[:4]={np.round(t[:4],3)} pred[:4]={np.round(p[:4],3)}')

show_reco_examples(model, train_loader, cfg)


## SECTION 12 — FINE-TUNING RUN


In [ ]:
ckpt_path = Path('./checkpoints/hybrid_mae_pretrained.pt')
ckpt_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), ckpt_path)

ft_model = HybridLorentzParTMAE(cfg).to(device)
ft_model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=False)

optf = make_optimizer(ft_model, cfg)
schf = make_scheduler(optf)
scalerf = torch.amp.GradScaler(device=device.type, enabled=amp_enabled)

ft_hist = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
for ep in range(cfg.FINETUNE_EPOCHS):
    tr = train_epoch_finetune(ft_model, train_loader, optf, scalerf, device, cfg)
    va, *_ = eval_finetune(ft_model, val_loader, device, cfg)
    schf.step()

    ft_hist['train_loss'].append(tr['loss'])
    ft_hist['train_acc'].append(tr['acc'])
    ft_hist['val_acc'].append(va['accuracy'])
    ft_hist['val_auc'].append(va['macro_auc_ovr'])

    print(f"[Finetune {ep+1}/{cfg.FINETUNE_EPOCHS}] train_loss={tr['loss']:.4f} train_acc={tr['acc']:.4f} val_acc={va['accuracy']:.4f} val_auc={va['macro_auc_ovr']:.4f}")


In [ ]:
plot_curves(ft_hist, 'Fine-tuning Curves')


In [ ]:
test_metrics, y_true, y_prob, y_pred = eval_finetune(ft_model, test_loader, device, cfg)
print('Test metrics:')
for k,v in test_metrics.items():
    print(f'  {k}: {v:.6f}')

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=[str(c) for c in class_names], zero_division=0))

plot_confusion(y_true, y_pred, class_names)
plot_rocs(y_true, y_prob, class_names)
print('Background rejection @50% signal efficiency (class 0):', bkg_rej_at_sig_eff(y_true, y_prob, sig_class=0, eff=0.5))


## SECTION 13 — ABLATION STUDY


In [ ]:
"""Ablations prepared:
- ParT-only
- Lorentz-only
- Naive concatenation-like (no gate)
- Gated fusion (proposed)
- with vs without MAE pretraining
- with vs without auxiliary mass regression
"""

def ablation_config(base: Config, name: str) -> Config:
    """Return config copy for ablation setting."""
    c = copy.deepcopy(base)
    if name == 'part_only':
        c.NUM_LAYERS_LORENTZ = 0
    elif name == 'lorentz_only':
        c.NUM_LAYERS_PART = 0
    elif name == 'naive_concat':
        c.USE_TOKEN_GATE = False; c.USE_EVENT_GATE = False
    elif name == 'gated_fusion':
        c.USE_TOKEN_GATE = True; c.USE_EVENT_GATE = True
    elif name == 'no_mass_aux':
        c.USE_AUX_MASS = False
    elif name == 'with_mass_aux':
        c.USE_AUX_MASS = True
    elif name in ['with_mae_pretrain','no_mae_pretrain']:
        pass
    else:
        raise ValueError(name)
    return c

def run_quick_ablations(cfg, train_loader, val_loader, names):
    """Run lightweight ablation loop for comparison-ready outputs."""
    rows = []
    for name in names:
        c = ablation_config(cfg, name)
        m = HybridLorentzParTMAE(c).to(device)
        o = make_optimizer(m, c)
        s = make_scheduler(o)
        g = torch.amp.GradScaler(device=device.type, enabled=amp_enabled)

        tr = train_epoch_finetune(m, train_loader, o, g, device, c)
        va, *_ = eval_finetune(m, val_loader, device, c)
        s.step()

        row = {'ablation': name, 'train_loss': tr['loss'], 'train_acc': tr['acc'], 'val_acc': va['accuracy'], 'val_auc': va['macro_auc_ovr']}
        rows.append(row)
        print(row)
    return pd.DataFrame(rows)

ablation_names = ['part_only','lorentz_only','naive_concat','gated_fusion','no_mass_aux','with_mass_aux']
ablation_df = run_quick_ablations(cfg, train_loader, val_loader, ablation_names)
ablation_df


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(ablation_df['ablation'], ablation_df['val_acc'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Validation Accuracy')
plt.title('Ablation Comparison (Quick Demo)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## SECTION 14 — INTERPRETABILITY / DIAGNOSTICS


In [ ]:
@torch.no_grad()
def inspect_gates(model, loader, n_batches=3):
    """Inspect event/token gate distributions and classwise means."""
    model.eval()
    ge, gt, yy = [], [], []
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        batch = to_device(batch, device)
        out = model(batch['x'], batch['p4'], batch['mask'], mode='finetune')
        ge.append(out['g_evt'].squeeze(-1).cpu().numpy())
        gt.append(out['g_tok'].mean(dim=(1,2)).cpu().numpy())
        yy.append(batch['y'].cpu().numpy())

    ge = np.concatenate(ge); gt = np.concatenate(gt); yy = np.concatenate(yy)
    print('Event gate summary:\n', pd.Series(ge).describe())
    print('Token gate summary:\n', pd.Series(gt).describe())

    plt.figure(figsize=(8,3))
    plt.hist(ge, bins=30, alpha=0.7, label='event gate')
    plt.hist(gt, bins=30, alpha=0.7, label='token gate mean')
    plt.legend(); plt.title('Gate distributions'); plt.show()

    df = pd.DataFrame({'y': yy, 'event_gate': ge, 'token_gate': gt})
    print('Mean gate by class:')
    print(df.groupby('y')[['event_gate','token_gate']].mean())

inspect_gates(ft_model, test_loader)


In [ ]:
def simple_saliency(model, batch):
    """Simple saliency magnitudes from input gradients."""
    model.eval()
    x = batch['x'].to(device).clone().detach().requires_grad_(True)
    p4, mask, y = batch['p4'].to(device), batch['mask'].to(device), batch['y'].to(device)

    out = model(x, p4, mask, mode='finetune')
    loss = F.cross_entropy(out['logits'], y)
    loss.backward()

    sal = x.grad.abs().mean(dim=-1).detach().cpu().numpy()
    vm = batch['mask'].numpy() > 0.5
    vals = sal[vm]
    print(pd.Series(vals).describe())
    plt.figure(figsize=(6,3)); plt.hist(vals, bins=40)
    plt.title('Input saliency (valid tokens)'); plt.show()

batch0 = next(iter(test_loader))
simple_saliency(ft_model, batch0)


In [ ]:
@torch.no_grad()
def reconstruction_failure_cases(model, loader, cfg, topk=5):
    """Find events with largest masked reconstruction error."""
    model.eval()
    errs = []
    for batch in loader:
        batch = to_device(batch, device)
        vis, tgt, rm, _ = build_mae_targets(batch['x'], batch['p4'], batch['mask'], cfg.MASK_RATIO)
        out = model(vis, batch['p4'], batch['mask'], mode='pretrain')
        per_evt = (((out['reco'] - tgt)**2).mean(dim=-1) * rm.float()).sum(dim=1)
        errs.extend(per_evt.cpu().numpy().tolist())

    errs = np.array(errs)
    idx = np.argsort(-errs)[:topk]
    print('Top failure indices:', idx)
    print('Errors:', errs[idx])

reconstruction_failure_cases(ft_model, test_loader, cfg)


## SECTION 15 — FINAL DISCUSSION

### What was implemented
A complete hybrid MAE notebook for JetClass-like ROOT data was implemented, including robust loading, feature engineering, hybrid architecture, training loops, evaluation, ablations, and diagnostics.

### Key improvement
The main proposal is **attention-gated hybrid fusion**, where branch trust is learned dynamically per event/token instead of using static fusion.

### Full-scale results to measure
On full JetClass data, report:
- top-1 accuracy and macro AUC,
- per-class ROC + confusion matrix,
- background rejection at fixed signal efficiency,
- with/without MAE pretraining,
- with/without auxiliary mass regression.

### GSoC alignment
This notebook supports a GSoC-level research path by combining physically motivated inductive biases with practical engineering and ablation-ready experimentation.

### Next steps for turning this notebook into a repository
Refactor into:
- `src/models/`
- `src/data/`
- `src/losses/`
- `src/trainers/`
- `configs/`
- `scripts/`


In [ ]:
print(json.dumps(asdict(cfg), indent=2))
